# Dolomite surface titration in NaCl  —  speciation, pH from charge balance, surface charge

After **Pokrovsky et al. (1999)**. This Colab notebook:

* writes the **law of mass action** for every aqueous complex,
* takes the **free ion concentrations** (H⁺, Ca²⁺, Mg²⁺, Na⁺, Cl⁻) as the unknowns, with carbonate set by a **fixed pCO₂**,
* solves the **mass balances** for Ca, Mg, Na, Cl,
* **calculates pH from the charge balance** (the measured pH is not used as input),
* lists the full **speciation for vessels A, B and C**,
* computes the **surface charge directly from the charge balance** — no surface complexation model — and plots it against pH.

Just press **Runtime ▸ Run all**. Every library used (numpy, scipy, matplotlib, pandas) ships with Colab.

## 1. Molar masses and unit conversion

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq, fsolve

# molar masses (g/mol)
mmNa, mmCl, mmCa, mmMg = 22.990, 35.453, 40.078, 24.305

def ppm_to_M(ppm, mm):
    """mg/L divided by g/mol -> mol/L"""
    return ppm / 1000.0 / mm

## 2. Equilibrium constants at 25 °C (log₁₀ K)

Reactions are written exactly as in the supplied TOUGHREACT / EQ3-6 database 
(complex on the left, basis species on the right); the number is log K of that dissociation.

In [ ]:
# carbonate system -- open system, pCO2 fixed
logKH = -1.468     # CO2(g)  = CO2(aq)      : [CO2aq] = 10**logKH * pCO2
logK1 = -6.345     # CO2(aq) + H2O = H+ + HCO3-
logK2 = -10.329    # HCO3-         = H+ + CO3^2-
logKw = -13.995    # H2O           = H+ + OH-

# metal-ligand complexes  (log K of the dissociation as written)
logK = {
    'CaCl':   0.696,   # CaCl+     = Ca2+ + Cl-
    'CaCO3':  7.002,   # CaCO3(aq) = Ca2+ + HCO3- - H+
    'CaHCO3':-1.047,   # CaHCO3+   = Ca2+ + HCO3-
    'CaOH':  12.850,   # CaOH+     = Ca2+ + H2O - H+
    'MgCl':   0.135,   # MgCl+     = Mg2+ + Cl-
    'MgCO3':  7.350,   # MgCO3(aq) = Mg2+ + HCO3- - H+
    'MgHCO3':-1.036,   # MgHCO3+   = Mg2+ + HCO3-
    'MgOH':  11.785,   # MgOH+     = Mg2+ + H2O - H+
    'NaCO3':  9.815,   # NaCO3-    = Na+ + HCO3- - H+
    'NaHCO3':-0.154,   # NaHCO3(aq)= Na+ + HCO3-
    'NaCl':   0.777,   # NaCl(aq)  = Na+ + Cl-
    'NaOH':  14.180,   # NaOH(aq)  = Na+ + H2O - H+
}

## 3. Law of mass action for every species

Given the free ion concentrations `h, cCa, cMg, cNa, cCl` and a fixed `pCO2`, 
each concentration below follows from its K. Water activity is taken as 1 
(concentration basis, no activity coefficients — see the note at the end).

In [ ]:
def all_species(h, cCa, cMg, cNa, cCl, pCO2):
    # carbonate fixed by pCO2 and h
    co2  = 10**logKH * pCO2
    hco3 = 10**logK1 * co2 / h        # CO2 + H2O = H+ + HCO3-
    co3  = 10**logK2 * hco3 / h       # HCO3-     = H+ + CO3^2-
    oh   = 10**logKw / h              # H2O       = H+ + OH-
    return {
        'H':   h,   'OH':  oh,  'CO2': co2, 'HCO3': hco3, 'CO3': co3,
        # calcium
        'Ca':     cCa,
        'CaCl':   cCa * cCl / 10**logK['CaCl'],
        'CaCO3':  cCa * hco3 / (h * 10**logK['CaCO3']),
        'CaHCO3': cCa * hco3 / 10**logK['CaHCO3'],
        'CaOH':   cCa / (h * 10**logK['CaOH']),
        # magnesium
        'Mg':     cMg,
        'MgCl':   cMg * cCl / 10**logK['MgCl'],
        'MgCO3':  cMg * hco3 / (h * 10**logK['MgCO3']),
        'MgHCO3': cMg * hco3 / 10**logK['MgHCO3'],
        'MgOH':   cMg / (h * 10**logK['MgOH']),
        # sodium
        'Na':     cNa,
        'NaCO3':  cNa * hco3 / (h * 10**logK['NaCO3']),
        'NaHCO3': cNa * hco3 / 10**logK['NaHCO3'],
        'NaCl':   cNa * cCl / 10**logK['NaCl'],
        'NaOH':   cNa / (h * 10**logK['NaOH']),
        # chloride
        'Cl':     cCl,
    }

## 4. Speciation solver

For a given pH and pCO₂, solve the four mass balances (Ca, Mg, Na, Cl) for the 
free ion concentrations, then return every species.

In [ ]:
def solve_speciation(caT, mgT, naT, clT, pH, pCO2):
    h = 10.0**(-pH)
    def residuals(x):
        cCa, cMg, cNa, cCl = [max(v, 1e-30) for v in x]
        s = all_species(h, cCa, cMg, cNa, cCl, pCO2)
        return [
            s['Ca'] + s['CaCl'] + s['CaCO3'] + s['CaHCO3'] + s['CaOH'] - caT,
            s['Mg'] + s['MgCl'] + s['MgCO3'] + s['MgHCO3'] + s['MgOH'] - mgT,
            s['Na'] + s['NaCO3'] + s['NaHCO3'] + s['NaCl'] + s['NaOH'] - naT,
            s['Cl'] + s['CaCl'] + s['MgCl'] + s['NaCl'] - clT,
        ]
    x0 = [max(caT, 1e-9), max(mgT, 1e-9), max(naT, 1e-9), max(clT, 1e-9)]
    x  = fsolve(residuals, x0, full_output=False)
    return all_species(h, *[max(v, 1e-30) for v in x], pCO2)

## 5. Charge balance and surface charge

`pos` and `neg` are the reactive-species charge sums requested. The indifferent 
electrolyte (free Na⁺ / Cl⁻) is left out on purpose: its imbalance is what the 
mineral surface carries.

```
pos = H + 2*Ca + 2*Mg + CaHCO3 + CaOH + MgHCO3 + MgOH
neg = OH + HCO3 + 2*CO3 + 2*NaCO3 + NaHCO3
```

The **calculated pH** is the pH where `sigma = pos - neg = 0` at fixed pCO₂.

In [ ]:
def sigma(s):
    pos = (s['H'] + 2*s['Ca'] + 2*s['Mg']
           + s['CaHCO3'] + s['CaOH'] + s['MgHCO3'] + s['MgOH'])
    neg = (s['OH'] + s['HCO3'] + 2*s['CO3']
           + 2*s['NaCO3'] + s['NaHCO3'])
    return pos - neg          # surface charge, eq/L

def pH_from_charge(caT, mgT, naT, clT, pCO2):
    f = lambda p: sigma(solve_speciation(caT, mgT, naT, clT, p, pCO2))
    # scan for the first sign change, then bracket with brentq
    ps = np.arange(2.0, 12.01, 0.25)
    vals = np.array([f(p) for p in ps])
    idx = np.where(np.sign(vals[:-1]) != np.sign(vals[1:]))[0]
    if len(idx) == 0:
        return np.nan
    k = idx[0]
    return brentq(f, ps[k], ps[k+1])

## 6. Fixed conditions and measured data

`pCO2` is the single knob (default atmospheric, 10^-3.5 atm). Columns of the 
data tables: measured pH, Cl (ppm), Na (ppm), Ca (ppm), Mg (ppm). Vessel B is the 
NaCl blank and has no Ca or Mg. Measured carbonate/alkalinity are not used here 
because the carbonate speciation is set by pCO₂.

In [ ]:
pCO2 = 10**-3.5      # atmospheric CO2 partial pressure (atm); change freely

# solid / solution ratios for the surface-charge normalisation
mass_solid = 6.0     # g of dolomite
vol_A = 0.100        # L in vessel A
vol_C = 0.200        # L in vessel C (100 mL A + 100 mL B)

# Vessel A : dolomite + NaCl, equilibrated
dataA = [
    [8.8, 39648, 23604.3, 30.7, 42.7],
    [8.8, 32751, 24657.7, 31.0, 46.6],
    [8.8, 36206, 26775.3, 35.0, 49.5],
    [8.9, 32807, 23506.8, 35.7, 45.0],
    [8.9, 48942, 27749.8, 29.4, 43.3],
    [8.9, 42883, 34199.3, 28.3, 41.2],
    [9.0, 38612, 29672.0, 28.2, 41.6],
    [9.0, 33835, 24947.5, 30.1, 43.8],
    [9.0, 30515, 21705.0, 30.6, 45.1],
]

# Vessel B : NaCl blank, pH pre-adjusted (Ca = Mg = 0). columns: pH, Cl, Na
dataB = [
    [1.8,  16777, 24113],
    [2.1,  17017, 32927],
    [2.1,  16827, 23180],
    [2.4,  16937, 53900],
    [5.5,  16838, 23718],
    [10.5, 16879, 21836],
    [10.6, 16854, 21302],
    [10.6, 16829, 21073],
    [10.8, 17022, 23484],
]

# Vessel C : mixture A + B after re-equilibration
dataC = [
    [7.0,  26800, 16953.4, 1325.3, 233.9],
    [7.2,  36300, 25729.3,  755.1, 180.4],
    [7.5,  32258, 23296.4,  560.6, 145.9],
    [7.5,  24901, 19593.1,  398.9, 114.2],
    [8.8,  24244, 16466.5,   37.2,  24.4],
    [9.8,  23699, 16183.2,   15.6,  13.6],
    [10.2, 21815, 14759.0,   16.3,   4.4],
    [10.2, 24118, 16450.8,   17.0,   3.6],
    [10.6, 22678, 15393.8,    9.8,   1.0],
]

## 7. Solve every sample

For each sample: convert ppm to mol/L, find the calculated pH from the charge 
balance, build the full speciation at that pH, and record the surface charge 
evaluated at the measured pH.

In [ ]:
def analyse_row(row, pCO2):
    if len(row) == 3:                      # vessel B (no Ca, Mg)
        pHm, clp, nap = row; cap = mgp = 0.0
    else:
        pHm, clp, nap, cap, mgp = row
    caT = ppm_to_M(cap, mmCa); mgT = ppm_to_M(mgp, mmMg)
    naT = ppm_to_M(nap, mmNa); clT = ppm_to_M(clp, mmCl)
    pHc   = pH_from_charge(caT, mgT, naT, clT, pCO2)
    sCalc = solve_speciation(caT, mgT, naT, clT, pHc,  pCO2) if np.isfinite(pHc) else None
    sMeas = solve_speciation(caT, mgT, naT, clT, pHm,  pCO2)
    return {'pHmeas': pHm, 'pHcalc': pHc, 'spec': sCalc,
            'Qmeas': sigma(sMeas)}      # surface charge at the measured pH, eq/L

resA = [analyse_row(r, pCO2) for r in dataA]
resB = [analyse_row(r, pCO2) for r in dataB]
resC = [analyse_row(r, pCO2) for r in dataC]

## 8. Measured vs calculated pH

In [ ]:
def pH_table(res, label):
    df = pd.DataFrame({
        'pH measured':  [r['pHmeas'] for r in res],
        'pH calculated':[round(r['pHcalc'], 2) if np.isfinite(r['pHcalc']) else np.nan for r in res],
        'Q at meas. pH (eq/L)': [r['Qmeas'] for r in res],
    })
    df.index = [f'S{i+1}' for i in range(len(res))]
    df.index.name = f'Vessel {label}'
    return df

for lab, res in [('A', resA), ('B', resB), ('C', resC)]:
    print(f'--- Vessel {lab}: measured vs calculated pH ---')
    display(pH_table(res, lab).style.format({'Q at meas. pH (eq/L)': '{:.3e}'}))

## 9. Speciation tables (at the calculated pH)

Species in rows, samples in columns, mol/L.

In [ ]:
species_order = ['H','OH','CO2','HCO3','CO3',
                 'Ca','CaCl','CaCO3','CaHCO3','CaOH',
                 'Mg','MgCl','MgCO3','MgHCO3','MgOH',
                 'Na','NaCO3','NaHCO3','NaCl','NaOH','Cl']

def speciation_table(res, label):
    cols = {}
    for i, r in enumerate(res):
        s = r['spec']
        cols[f'S{i+1}'] = [ (s[k] if s is not None else np.nan) for k in species_order ]
    df = pd.DataFrame(cols, index=species_order)
    df.index.name = f'Vessel {label} (mol/L)'
    return df

for lab, res in [('A', resA), ('B', resB), ('C', resC)]:
    print(f'--- Vessel {lab}: speciation at the calculated pH ---')
    display(speciation_table(res, lab).style.format('{:.3e}'))

## 10. Surface charge

`sigma(pH)` is the reactive charge imbalance the surface must carry. It crosses 
zero at the calculated pH (no surface interaction) and turns negative / positive 
when the real (measured) pH sits above / below it. Values are normalised per gram 
of dolomite (6 g of solid; 100 mL in A, 200 mL in C).

In [ ]:
def mean_totals(data):
    rows = []
    for r in data:
        if len(r) == 3:
            pHm, cl, na = r; ca = mg = 0.0
        else:
            pHm, cl, na, ca, mg = r
        rows.append([ppm_to_M(ca, mmCa), ppm_to_M(mg, mmMg),
                     ppm_to_M(na, mmNa), ppm_to_M(cl, mmCl)])
    return np.mean(rows, axis=0)

def sigma_curve(tot, p):
    caT, mgT, naT, clT = tot
    return sigma(solve_speciation(caT, mgT, naT, clT, p, pCO2))

def per_gram(Q, vol):
    return Q * vol / mass_solid          # mol charge per g of solid

tA, tC = mean_totals(dataA), mean_totals(dataC)
pH_grid = np.linspace(6, 11, 120)
curveA = [per_gram(sigma_curve(tA, p), vol_A)*1000 for p in pH_grid]  # mmol/g
curveC = [per_gram(sigma_curve(tC, p), vol_C)*1000 for p in pH_grid]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) titration curve of the mean composition
ax[0].axhline(0, color='0.6', lw=0.8)
ax[0].plot(pH_grid, curveA, label='Vessel A (mean)', color='tab:red')
ax[0].plot(pH_grid, curveC, label='Vessel C (mean)', color='tab:blue')
ax[0].set_xlabel('pH (calculated)'); ax[0].set_ylabel('surface charge  (mmol / g)')
ax[0].set_title('Surface charge from the charge balance  (pCO2 fixed)')
ax[0].legend(); ax[0].grid(alpha=0.3)

# (b) each sample at its measured pH
ptsA = [(r['pHmeas'], per_gram(r['Qmeas'], vol_A)*1000) for r in resA]
ptsC = [(r['pHmeas'], per_gram(r['Qmeas'], vol_C)*1000) for r in resC]
ax[1].axhline(0, color='0.6', lw=0.8)
ax[1].scatter(*zip(*ptsA), label='Vessel A', color='tab:red')
ax[1].scatter(*zip(*ptsC), label='Vessel C', color='tab:blue')
ax[1].set_xlabel('pH (measured)'); ax[1].set_ylabel('surface charge  (mmol / g)')
ax[1].set_title('Surface charge at the measured pH')
ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Notes

1. Concentrations are used directly (activity coefficients = 1). At the very high 
ionic strength of these NaCl solutions a Davies / SIT / Pitzer correction would 
shift the numbers; add gamma factors inside `all_species` if needed.
2. Carbonate is fixed by pCO₂. To use the measured alkalinity instead, replace 
the `hco3` / `co3` lines in `all_species` with a carbonate mass balance.
3. The surface charge here is purely a charge-balance quantity — no surface 
complexation constants are fitted or assumed.
4. Vessel B is the blank: its calculated pH reflects only the water / carbonate 
system because the strong acid or base used to preset its pH is not one of the 
species in the balance.